Data Augmentation


In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image

from tensorflow.keras.preprocessing.image import ImageDataGenerator


csv_path = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train.csv"

train_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train\train"

output_dir = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented"

output_csv = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented.csv"


df = pd.read_csv(csv_path)

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)


os.makedirs(output_dir, exist_ok=True)

new_rows = []

for _, row in df.iterrows():

    img_id = str(row["Id"])
    label = str(row["Category"])

    img_path = os.path.join(
        train_dir,
        label,
        img_id + ".png"
    )

    img = Image.open(img_path).convert("L")
    img_array = np.array(img)

    class_output_dir = os.path.join(output_dir, label)
    os.makedirs(class_output_dir, exist_ok=True)

    original_filename = img_id + ".png"

    original_output_path = os.path.join(
        class_output_dir,
        original_filename
    )

    img.save(original_output_path)

    new_rows.append({
        "Id": img_id,
        "Category": int(label)
    })


    img_array = img_array.reshape((1, 32, 32, 1))

    aug_iter = datagen.flow(img_array, batch_size=1)

    for i in range(1):

        aug_img = next(aug_iter)[0].astype(np.uint8)

        aug_img = aug_img.reshape(32, 32)

        aug_pil = Image.fromarray(aug_img)

        aug_filename = f"{img_id}_aug{i}.png"

        aug_output_path = os.path.join(
            class_output_dir,
            aug_filename
        )

        aug_pil.save(aug_output_path)

        new_rows.append({
            "Id": f"{img_id}_aug{i}",
            "Category": int(label)
        })


new_df = pd.DataFrame(new_rows)

new_df.to_csv(output_csv, index=False)

print("DONE")
print("Augmented dataset saved to:")
print(output_dir)

print("\nNew CSV saved to:")
print(output_csv)

print("\nTotal images:", len(new_df))

DONE
Augmented dataset saved to:
C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented

New CSV saved to:
C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_augmented.csv

Total images: 34000


Submission csv cleaner


In [ ]:
import pandas as pd

df = pd.read_csv('C:\\Users\\alijo\\OneDrive\\Desktop\\IVP project\\IVP-Group32\\submission_augmented.csv')

df['Id'] = df['Id'].str.replace('.png', '', regex=False)

df.to_csv('C:\\Users\\alijo\\OneDrive\\Desktop\\IVP project\\IVP-Group32\\submission_augmented.csv', index=False)

Noising/Distortion for train split

In [ ]:
import os
from PIL import Image
import torchvision.transforms as T
import csv


input_root = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train\train"
output_root = r"C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_V4"


transform = T.Compose([
    T.RandomRotation(10),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.95, 1.05), shear=5),
    T.GaussianBlur(kernel_size=3)
])


def create_augmented_dataset():
    rows = []

    for class_name in sorted(os.listdir(input_root)):
        class_input_dir = os.path.join(input_root, class_name)
        class_output_dir = os.path.join(output_root, class_name)

        if not os.path.isdir(class_input_dir):
            continue

        os.makedirs(class_output_dir, exist_ok=True)
        print(f"Processing class: {class_name}")

        for filename in sorted(os.listdir(class_input_dir)):
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            img_path = os.path.join(class_input_dir, filename)
            img = Image.open(img_path).convert("L")
            stem = os.path.splitext(filename)[0]

            img.save(os.path.join(class_output_dir, f"{stem}.png"))
            rows.append([stem, class_name])

            aug_img = transform(img)
            aug_img.save(os.path.join(class_output_dir, f"{stem}_aug.png"))
            rows.append([f"{stem}_aug", class_name])

    csv_path = os.path.join(output_root, "train_V3_labels.csv")
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Id', 'Category'])
        writer.writerows(rows)

    print(f"DONE — CSV saved to {csv_path} with {len(rows)} entries")


create_augmented_dataset()

Processing class: 0
Processing class: 1
Processing class: 2
Processing class: 3
Processing class: 4
Processing class: 5
Processing class: 6
Processing class: 7
Processing class: 8
Processing class: 9
DONE — CSV saved to C:\Users\alijo\OneDrive\Desktop\iivp-2026-challenge\train_V4\train_V3_labels.csv with 34000 entries
